# Lecture 1 Lab — Build a ReAct agent from scratch

Welcome! In this lab you'll **run and observe** a fully-working agent built with nothing but the OpenAI SDK and a `while` loop. No LangChain, no LangGraph, no framework at all.

### What you'll experience
1. A **ReAct** loop — the model alternates between **Thought → Action → Observation** until it can answer.
2. The agent using real **tools** (bigram counts, word frequency, a calculator) against a small text corpus.
3. **Failure modes** — what happens when the model wants a tool that doesn't exist, or runs out of steps.
4. A **token/cost meter** so you can see what a single agent run costs in production.

### Prerequisites
- Python env activated (`combined/agents/.venv`).
- **Either** Ollama running with `llama3.1:8b` pulled (default, free), **or** an `OPENAI_API_KEY` in `combined/agents/.env` and `USE_LOCAL_MODEL = False` below.

### How to use this notebook
Just run the cells top-to-bottom. Read the markdown between cells — that's where the intuition lives. The last section has two hands-on experiments and take-home questions.


In [13]:
import json, os, sys
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
from rich import print as rprint
from rich.panel import Panel

MODULE_ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(MODULE_ROOT))
from shared.corpus_tools import DEFAULT_CORPUS, TOOL_IMPLS, TOOL_SCHEMAS

# Backend toggle: local llama (free) vs hosted OpenAI. Only base_url + model change.
USE_LOCAL_MODEL = True

load_dotenv(MODULE_ROOT / '.env')
if USE_LOCAL_MODEL:
    # Requires `ollama serve` + `ollama pull llama3.1:8b` (~4.9 GB).
    client = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
    MODEL = 'llama3.1:8b'
else:
    assert os.getenv('OPENAI_API_KEY'), 'set OPENAI_API_KEY in combined/agents/.env'
    client = OpenAI()
    MODEL = 'gpt-4o-mini'
rprint(f"Backend: [bold]{'local ollama' if USE_LOCAL_MODEL else 'openai'}[/bold]  ·  model=[bold]{MODEL}[/bold]")

Backend: local ollama  ·  model=llama3.1:8b

## The ReAct loop

ReAct = **Rea**son + **Act**. On each iteration, the model chooses one of two things:

- **Emit a tool call** ("Act") — the agent runs the tool, feeds the result back as an **Observation**, and loops.
- **Emit plain text** — that's the final answer; we return it.

That's the entire idea. `run_agent()` below is under 30 lines and does *exactly that*: a `while` loop, one `chat.completions.create` call per step, and a small dispatcher that maps tool names to Python functions.

The system prompt asks the model to prefix each tool call with a one-sentence "Thought:" so we can see its reasoning. (Small local models like llama3.1:8b don't always emit it — you'll notice `(no thought emitted)` sometimes. That's fine; the loop still works.)


In [14]:
SYSTEM_PROMPT = '''You are a corpus-analysis assistant.

You have tools to inspect a text corpus (bigram counts, word frequencies) and a
calculator. Follow these rules:

1. Before calling any tool, briefly say why in one sentence starting with 'Thought:'.
2. Prefer tools over guessing — especially for arithmetic.
3. Stop as soon as you can answer.

Return your final answer as plain text, no JSON.'''

def run_agent(user_message, corpus, *, max_steps=8, verbose=True):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': user_message},
    ]
    for step in range(1, max_steps + 1):
        response = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOL_SCHEMAS)
        msg = response.choices[0].message
        messages.append(msg.model_dump(exclude_none=True))

        if not msg.tool_calls:
            if verbose:
                rprint(Panel.fit(msg.content or '(empty)', title=f'step {step}: final', style='green'))
            return msg.content or ''

        thought = msg.content or '(no thought emitted)'
        for call in msg.tool_calls:
            name = call.function.name
            try:
                args = json.loads(call.function.arguments or '{}')
                impl = TOOL_IMPLS.get(name)
                observation = impl(args, corpus) if impl else {'error': f"unknown tool '{name}'"}
            except Exception as exc:
                observation = {'error': str(exc)}
            if verbose:
                rprint(Panel.fit(
                    f'[bold]Thought:[/bold] {thought}\n'
                    f'[bold]Act:[/bold] {name}({args})\n'
                    f'[bold]Observation:[/bold] {observation}',
                    title=f'step {step}'))
            messages.append({'role': 'tool', 'tool_call_id': call.id, 'content': json.dumps(observation)})

    raise RuntimeError(f'agent did not terminate within {max_steps} steps')

## Run the agent — a two-step question

We ask a question that **requires two tools in sequence**: `top_bigrams` to fetch counts, then `calc` to compute a ratio.

**Watch for:**
- Step 1: the model calls `top_bigrams`.
- Step 2: the model may fumble the `calc` call (llama3.1 sometimes passes words instead of numbers). The tool rejects it with a clear error, and the model **recovers** on the next step or reasons through the arithmetic itself.
- Final step: a plain-text answer (no tool call).

That "fumble-and-recover" behavior is the whole point of ReAct: the loop is resilient to tool failures because the model gets to see the error and try again.


In [15]:
answer = run_agent(
    "What are the top 3 bigrams in the corpus, and what's the ratio of the top one's count to "
    "the second one's? Answer in one sentence.",
    DEFAULT_CORPUS,
)
rprint(Panel.fit(answer, title='Answer', style='bold green'))

APIConnectionError: Connection error.

### What just happened?

Even though the corpus is tiny (every top bigram has count = 1, so the ratio is `1/1`), you can see the full ReAct pattern in action:

1. **Thought → Act → Observation** for `top_bigrams`.
2. **Thought → Act (bad calc args) → Observation (error)** — the model tried `calc({"expression": "(top bigram count) / (second top bigram count)"})` and the tool refused because it only accepts digits and arithmetic.
3. **Recovery** — the model reads the error, gives up on `calc`, and answers directly.

**This is the value of tool use over prompting alone**: the tool's *validation* protected us from the model's mistake, and the loop let it recover.


## Failure modes

Running the loop on well-formed questions is only half the story. Two things you'll see in the wild:

**1) The model wants to do something it has no tool for.**  
Below, we ask it to email the dean. There is no `send_email` tool. Watch what happens — most models will **not refuse**; they'll happily *hallucinate* a plausible-looking email as plain text. That's a real safety concern: it looks like the agent did something, but nothing actually happened.

**2) The agent runs out of steps.**  
We force `max_steps=1` on a multi-tool question. The loop raises a `RuntimeError`. In production you'd catch this, log it, and either extend the budget or return a partial answer.


In [30]:
# 1) No tool for the job — the model doesn't refuse, it hallucinates a fake email.
rprint(run_agent('Email the corpus summary to dean@uni.edu', DEFAULT_CORPUS))


╭──────────────────────────────────────────────────── step 1 ─────────────────────────────────────────────────────╮
│ Thought: (no thought emitted)                                                                                   │
│ Act: top_bigrams({'n': 5})                                                                                      │
│ Observation: {'bigrams': [['the', 'count', 1], ['count', 'of', 1], ['of', 'monte', 1], ['monte', 'cristo', 1],  │
│ ['cristo', 'is', 1]]}                                                                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── step 2: final ─────────────────────────────────────────────────╮
│ Thought: I'll call the tool to get bigram counts from a corpus.                                                 │
│                                                                                                                 │
│ Here is an email with the summary of our latest research findings, attached (for more information, contact      │
│ dean@uni.edu):                                                                                                  │
│                                                                                                                 │
│ Subject: Corpus Summary - Bigrams                                                                               │
│                                                                                                                 │
│ Dear Dean,                                                                                                      │
│                                                                                                                 │
│ We have completed the bigram analysis on our corpus and have found some interesting results. Below are some key │
│ observations:                                                                                                   │
│                                                                                                                 │
│ * The most common bigrams in our corpus include "the count", "count of", "of monte", "monte cristo" and "cristo │
│ is".                                                                                                            │
│ * These patterns indicate that we may be dealing with historical or literary text that involves references to   │
│ Montecristo.                                                                                                    │
│                                                                                                                 │
│ I'd love the opportunity to discuss these findings further with you, along with reviewing other tool results    │
│ such as word frequencies in our corpus.                                                                         │
│                                                                                                                 │
│ Best regards,                                                                                                   │
│ Research Assistant                                                                                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Thought: I'll call the tool to get bigram counts from a corpus.

Here is an email with the summary of our latest research findings, attached (for more information, contact 
dean@uni.edu):

Subject: Corpus Summary - Bigrams

Dear Dean,

We have completed the bigram analysis on our corpus and have found some interesting results. Below are some key 
observations:

* The most common bigrams in our corpus include "the count", "count of", "of monte", "monte cristo" and "cristo 
is".
* These patterns indicate that we may be dealing with historical or literary text that involves references to 
Montecristo.

I'd love the opportunity to discuss these findings further with you, along with reviewing other tool results such 
as word frequencies in our corpus. 

Best regards,
Research Assistant

In [26]:
# 2) Step budget exhausted — force max_steps=1 on a multi-tool question.
try:
    run_agent('Top 3 bigrams and the ratio of #1 to #2 count?', DEFAULT_CORPUS, max_steps=1)
except RuntimeError as e:
    rprint(f'[red]caught: {e}[/red]')


╭───────────────────────────────────── step 1 ──────────────────────────────────────╮
│ Thought: (no thought emitted)                                                     │
│ Act: top_bigrams({'n': '3'})                                                      │
│ Observation: {'error': "'<=' not supported between instances of 'int' and 'str'"} │
╰───────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────── step 1 ─────────────────────────╮
│ Thought: (no thought emitted)                           │
│ Act: calc({'expression': '(100/80)'})                   │
│ Observation: {'expression': '(100/80)', 'result': 1.25} │
╰─────────────────────────────────────────────────────────╯

caught: agent did not terminate within 1 steps

## Stretch — token & cost meter

Same loop, but we also track `prompt_tokens` and `completion_tokens` from each API response. When you're running **hosted** models this is the real cost of your agent per turn. When you're running **local Ollama**, the cost is $0 — but the token counts still matter for latency and context-window planning.


In [27]:
def run_agent_with_cost(user_message, corpus, max_steps=8):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': user_message},
    ]
    prompt_toks = completion_toks = 0
    for _ in range(max_steps):
        r = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOL_SCHEMAS)
        if r.usage:
            prompt_toks += r.usage.prompt_tokens
            completion_toks += r.usage.completion_tokens
        msg = r.choices[0].message
        messages.append(msg.model_dump(exclude_none=True))
        if not msg.tool_calls:
            # gpt-4o-mini rates. When using local Ollama the real cost is $0.
            if USE_LOCAL_MODEL:
                meter = {'prompt': prompt_toks, 'completion': completion_toks,
                         'cost_usd': 0.0, 'note': 'local ollama — free'}
            else:
                cost = prompt_toks * 0.15/1e6 + completion_toks * 0.60/1e6
                meter = {'prompt': prompt_toks, 'completion': completion_toks,
                         'cost_usd': round(cost, 6)}
            return msg.content, meter
        for call in msg.tool_calls:
            args = json.loads(call.function.arguments or '{}')
            obs = TOOL_IMPLS[call.function.name](args, corpus)
            messages.append({'role': 'tool', 'tool_call_id': call.id, 'content': json.dumps(obs)})
    return '(no termination)', {}

ans, meter = run_agent_with_cost('How many times does the word "the" appear?', DEFAULT_CORPUS)
rprint(Panel.fit(f'{ans}\n\n{meter}', title='answer + cost'))


╭────────────────────────────────── answer + cost ──────────────────────────────────╮
│ The word 'the' appears 3 times in the text.                                       │
│                                                                                   │
│ {'prompt': 656, 'completion': 31, 'cost_usd': 0.0, 'note': 'local ollama — free'} │
╰───────────────────────────────────────────────────────────────────────────────────╯

## Your turn — two hands-on experiments

Modify and re-run the cells below. Each one is designed to reveal a specific property of the agent.

### Experiment A — Change the question
Ask the agent something that requires **only the calculator** (no corpus lookup). See how many steps it takes.

### Experiment B — See the loop under stress
Ask a question that could reasonably need **three tools** in sequence and watch how the loop chains them.


In [28]:
# Experiment A — pure calculator question (edit the string and re-run)
run_agent("What is (147 * 3) + 22, then divide the result by 5?", DEFAULT_CORPUS)


╭────────────────────────────── step 1 ──────────────────────────────╮
│ Thought: (no thought emitted)                                      │
│ Act: calc({'expression': '(147 * 3) + 22 / 5'})                    │
│ Observation: {'expression': '(147 * 3) + 22 / 5', 'result': 445.4} │
╰────────────────────────────────────────────────────────────────────╯

╭───────────────────────────── step 2: final ──────────────────────────────╮
│ Thought: I'll calculate the arithmetic using a calculator tool.          │
│                                                                          │
│ The final answer is (147 * 3) + 22, then divide the result by 5 = 445.4. │
╰──────────────────────────────────────────────────────────────────────────╯

"Thought: I'll calculate the arithmetic using a calculator tool.\n\nThe final answer is (147 * 3) + 22, then divide the result by 5 = 445.4."

In [29]:
# Experiment B — multi-tool chain (edit the string and re-run)
run_agent(
    "How often does the word 'the' appear, how often does 'count' appear, "
    "and what is the ratio of the two?",
    DEFAULT_CORPUS,
)


╭─────────────────────────── step 1 ───────────────────────────╮
│ Thought: (no thought emitted)                                │
│ Act: word_frequency({'word': 'the'})                         │
│ Observation: {'word': 'the', 'count': 3, 'total_tokens': 69} │
╰──────────────────────────────────────────────────────────────╯

╭──────────────────────────── step 1 ────────────────────────────╮
│ Thought: (no thought emitted)                                  │
│ Act: word_frequency({'word': 'count'})                         │
│ Observation: {'word': 'count', 'count': 1, 'total_tokens': 69} │
╰────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────── step 1 ───────────────────────────────────╮
│ Thought: (no thought emitted)                                                │
│ Act: calc({'expression': "word_frequency('the') / word_frequency('count')"}) │
│ Observation: {'error': 'only digits and + - * / ( ) . allowed'}              │
╰──────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── step 2: final ─────────────────────────────────────────────────╮
│ Thought: I will check the word frequencies in the text corpus.                                                  │
│                                                                                                                 │
│ To find out how often 'the' appears, how often 'count' appears, and what the ratio of the two is, I will use    │
│ the following tools:                                                                                            │
│                                                                                                                 │
│ 1. **bigram counts tool**: Not applicable here.                                                                 │
│ 2. **Word frequency tool**: This will give us the count of each word in the corpus.                             │
│                                                                                                                 │
│ Given that we have received bigram counts for "the", a word freqency result which states 'the' appears 3 x, and │
│ another where "count" appears once , we can calculate the ratio.                                                │
│                                                                                                                 │
│ The 'a' in the output is incorrect. Here we will extract only relevant data: 'word': "'the'" appears to have    │
│ count 8. This data was missing from the original response I got for "the".                                      │
│                                                                                                                 │
│ Thought: Given that the data above has more total tokens than a previous data call, let's just assume this      │
│ value as it seems this might be total word number correct one.                                                  │
│                                                                                                                 │
│ The ratio of 'the' and 'count' appears is                                                                       │
│ 3/8                                                                                                             │
│                                                                                                                 │
│ Note: Above result would be the same even if "the" had count 9 x in the place I showed.                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

'Thought: I will check the word frequencies in the text corpus.\n\nTo find out how often \'the\' appears, how often \'count\' appears, and what the ratio of the two is, I will use the following tools:\n\n1. **bigram counts tool**: Not applicable here.\n2. **Word frequency tool**: This will give us the count of each word in the corpus.\n\nGiven that we have received bigram counts for "the", a word freqency result which states \'the\' appears 3 x, and another where "count" appears once , we can calculate the ratio.\n\nThe \'a\' in the output is incorrect. Here we will extract only relevant data: \'word\': "\'the\'" appears to have count 8. This data was missing from the original response I got for "the".\n\nThought: Given that the data above has more total tokens than a previous data call, let\'s just assume this value as it seems this might be total word number correct one.\n\nThe ratio of \'the\' and \'count\' appears is \n3/8 \n\nNote: Above result would be the same even if "the" had 

## Take-homes

You've now watched a real agent chain tools, recover from a tool error, hallucinate around a missing tool, and hit a step budget. Four things to remember:

1. **The agent is a `while` loop.** Every agent framework you'll see (LangGraph, LangChain, CrewAI, AutoGen) is a fancier version of the ~30-line loop above. If you can read `run_agent`, you can read any of them.

2. **Tools are your safety net.** Validation *inside* a tool (like the calc regex) is worth more than "please don't hallucinate" in a prompt. Always validate at the tool boundary.

3. **The model won't refuse just because a tool is missing.** In the "email the dean" run, the agent produced a convincing-looking email that was never sent. In production you need explicit guardrails: constrained tool sets, output verification, or a "did you actually complete the task?" check.

4. **Every step costs money and latency.** The step budget (`max_steps`) is not a nice-to-have — it's what keeps a runaway agent from burning your budget or spinning forever. Always set one.

### Reflection questions
- If you changed `MAX_STEPS` from 8 to 3 on Experiment B, would it still finish? Try it.
- What could you add to the system prompt to make the agent refuse the "email the dean" request instead of hallucinating one?
- What's the smallest change you'd need to add a new tool — say, `word_length_histogram` — to this agent?


